# EMI & Loan Advisor Agent - Demo Notebook

This notebook demonstrates the agent's **plan-act loop** with **tool calling** and **memory persistence**.

The agent:
1. **PLANS** - Decides which tools to call based on user intent
2. **ACTS** - Executes tools in sequence
3. **OBSERVES** - Uses tool results to formulate response
4. **REMEMBERS** - Stores conversation in file-based memory

In [ ]:
import sys
import io
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')

from agent.core import create_agent

agent = create_agent()
print(f"Session ID: {agent.session_id}")
print(f"Memory file: data/{agent.session_id}.json")

## Goal 1: Calculate EMI for a Home Loan

**User**: "Calculate EMI for 50 lakhs home loan at 9% for 20 years"

In [ ]:
response = agent.run('Calculate EMI for 50 lakhs home loan at 9% for 20 years')
print(response)

# Show the internal trace
turn = agent.memory.turns[-1]
print("\n=== AGENT TRACE ===")
print(f"PLAN: {turn.plan}")
print(f"TOOL CALLS:")
for tc in turn.tool_calls:
    print(f"  - {tc['tool']}({tc['args']})")
    print(f"    Reason: {tc['reason']}")
print(f"TOOL RESULTS:")
for tr in turn.tool_results:
    print(f"  - {tr['tool_name']}: success={tr['success']}")

## Goal 2: Check Eligibility (Uses Memory from Previous Turn)

**User**: "My income is 1.5L, expenses 50K, age 30. Am I eligible for a home loan?"

The agent remembers the previous EMI calculation context.

In [ ]:
response = agent.run('My income is 1.5L, expenses 50K, age 30. Am I eligible for a home loan?')
print(response)

turn = agent.memory.turns[-1]
print("\n=== AGENT TRACE ===")
print(f"PLAN: {turn.plan}")
print(f"TOOL CALLS:")
for tc in turn.tool_calls:
    print(f"  - {tc['tool']}({tc['args']})")
    print(f"    Reason: {tc['reason']}")

## Goal 3: Multi-Step - Prepayment Analysis (Uses Previous Loan Context)

**User**: "What if I prepay 5L after 2 years?"

The agent reuses the loan parameters from the first EMI calculation (50L, 9%, 20yr) instead of asking again.

In [ ]:
response = agent.run('What if I prepay 5L after 2 years?')
print(response)

turn = agent.memory.turns[-1]
print("\n=== AGENT TRACE ===")
print(f"PLAN: {turn.plan}")
print(f"TOOL CALLS:")
for tc in turn.tool_calls:
    print(f"  - {tc['tool']}({tc['args']})")
    print(f"    Reason: {tc['reason']}")

## Goal 4: Loan Comparison

**User**: "Compare 50 lakhs at 9% vs 50 lakhs at 8.5% for 20 years"

In [ ]:
response = agent.run('Compare 50 lakhs at 9% vs 50 lakhs at 8.5% for 20 years')
print(response)

turn = agent.memory.turns[-1]
print("\n=== AGENT TRACE ===")
print(f"PLAN: {turn.plan}")
print(f"TOOL CALLS:")
for tc in turn.tool_calls:
    print(f"  - {tc['tool']}({tc['args']})")
    print(f"    Reason: {tc['reason']}")

## Goal 5: Affordability Check

**User**: "How much home loan can I afford with 2L income and 60K expenses?"

In [ ]:
response = agent.run('How much home loan can I afford with 2L income and 60K expenses?')
print(response)

turn = agent.memory.turns[-1]
print("\n=== AGENT TRACE ===")
print(f"PLAN: {turn.plan}")
print(f"TOOL CALLS:")
for tc in turn.tool_calls:
    print(f"  - {tc['tool']}({tc['args']})")
    print(f"    Reason: {tc['reason']}")

## Memory Persistence Demo

The conversation is saved to `data/{session_id}.json` and survives restarts.

In [ ]:
# Show memory contents
import json
with open(f'data/{agent.session_id}.json', 'r') as f:
    memory = json.load(f)

print(f"Session ID: {memory['session_id']}")
print(f"Turns: {len(memory['turns'])}")
print(f"User Profile: {memory['user_profile']}")

for i, turn in enumerate(memory['turns']):
    print(f"\n--- Turn {i+1} ---")
    print(f"User: {turn['user_input'][:80]}...")
    print(f"Tools: {[tc['tool'] for tc in turn['tool_calls']]}")

## Export Report (PDF/Excel)

Generate a comprehensive report of all calculations.

In [ ]:
export_data = agent.export_data()

from agent.export import generate_reports
reports = generate_reports(export_data, ['pdf', 'excel'])

print(f"PDF Report: {len(reports.get('pdf', b''))} bytes")
print(f"Excel Report: {len(reports.get('excel', b''))} bytes")

# Save files
with open(f'report_{agent.session_id}.pdf', 'wb') as f:
    f.write(reports['pdf'])
with open(f'report_{agent.session_id}.xlsx', 'wb') as f:
    f.write(reports['excel'])
print("Reports saved!")